# Healthy Start — Python Detection Server (Colab)

Runs the FastAPI dental detection server on Colab with a public tunnel.
The public URL connects to the Vercel frontend at `levelupwrestlingapp.com/hs/loop`.

**Requirements:** GPU runtime (T4 recommended). Go to Runtime > Change runtime type > T4 GPU.

## 1. Clone Repo & Install Dependencies

In [ ]:
# Clone the repo
!git clone https://github.com/sportsmockery/LevelUp.git /content/levelup
%cd /content/levelup/python

# Install dependencies
!pip install -q ultralytics>=8.3.0 fastapi>=0.115.0 uvicorn>=0.34.0 \
  python-multipart>=0.0.18 Pillow>=11.0.0 segment-anything>=1.0 \
  pyngrok torch torchvision opencv-python-headless

print('\n--- Dependencies installed ---')

## 2. Download Models

**SAM model** downloads automatically from Meta.  
**YOLO model** — upload `yolov12s_010826.pt` using the file panel on the left,  
or place it in Google Drive and mount below.

In [ ]:
import os, urllib.request, pathlib

MODEL_DIR = pathlib.Path('/content/levelup/python')

# --- SAM vit_b (358 MB) ---
SAM_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
SAM_PATH = MODEL_DIR / 'sam_vit_b_01ec64.pth'

if not SAM_PATH.exists():
    print('Downloading SAM vit_b model (358 MB)...')
    urllib.request.urlretrieve(SAM_URL, str(SAM_PATH))
    print(f'SAM model saved: {SAM_PATH} ({SAM_PATH.stat().st_size / 1e6:.0f} MB)')
else:
    print(f'SAM model already exists: {SAM_PATH}')

# --- YOLO model (18 MB) ---
YOLO_PATH = MODEL_DIR / 'yolov12s_010826.pt'

if not YOLO_PATH.exists():
    # Try Google Drive mount
    gdrive_path = pathlib.Path('/content/drive/MyDrive/models/yolov12s_010826.pt')
    if gdrive_path.exists():
        import shutil
        shutil.copy(str(gdrive_path), str(YOLO_PATH))
        print(f'YOLO model copied from Google Drive: {YOLO_PATH}')
    else:
        print('\n' + '='*60)
        print('ACTION REQUIRED: Upload yolov12s_010826.pt')
        print('='*60)
        print('Options:')
        print('  1. Drag yolov12s_010826.pt into the Files panel (left sidebar)')
        print('     Then run: !cp /content/yolov12s_010826.pt /content/levelup/python/')
        print('  2. Mount Google Drive and place it at:')
        print(f'     {gdrive_path}')
        print('='*60 + '\n')
else:
    print(f'YOLO model already exists: {YOLO_PATH}')

# Verify
print(f'\nSAM ready: {SAM_PATH.exists()} ({SAM_PATH.stat().st_size / 1e6:.0f} MB)' if SAM_PATH.exists() else 'SAM: MISSING')
print(f'YOLO ready: {YOLO_PATH.exists()} ({YOLO_PATH.stat().st_size / 1e6:.0f} MB)' if YOLO_PATH.exists() else 'YOLO: MISSING')

## 2b. (Optional) Upload YOLO model manually

If the YOLO model wasn't found above, upload it with the Files panel then run this cell:

In [ ]:
# Run this after uploading yolov12s_010826.pt to /content/
import shutil, pathlib
src = pathlib.Path('/content/yolov12s_010826.pt')
dst = pathlib.Path('/content/levelup/python/yolov12s_010826.pt')
if src.exists() and not dst.exists():
    shutil.copy(str(src), str(dst))
    print(f'Copied to {dst}')
elif dst.exists():
    print('YOLO model already in place')
else:
    print('Upload yolov12s_010826.pt to /content/ first')

## 3. Create Data Directories

In [ ]:
import pathlib
# Create directories the server/loop expects
for d in ['data/raw/train/images', 'data/labeling_queue', 'runs']:
    pathlib.Path(f'/content/levelup/python/{d}').mkdir(parents=True, exist_ok=True)
print('Data directories created')

## 4. Start Public Tunnel (ngrok)

Get a free ngrok auth token at https://dashboard.ngrok.com/get-started/your-authtoken  
Paste it below. This gives you a public HTTPS URL that Vercel can reach.

In [ ]:
NGROK_AUTH_TOKEN = ''  # <-- PASTE YOUR TOKEN HERE

from pyngrok import ngrok

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print('WARNING: No ngrok token set. Get one free at https://dashboard.ngrok.com')
    print('Without a token, the tunnel may be rate-limited.\n')

# Open tunnel to port 8100
public_url = ngrok.connect(8100, 'http')
print('='*60)
print(f'PUBLIC URL: {public_url}')
print('='*60)
print(f'\nSet this in Vercel environment variables:')
print(f'  HS_DETECTION_URL = {public_url}')
print(f'\nOr run locally:')
print(f'  vercel env add HS_DETECTION_URL')
print(f'  (paste: {public_url})')

## 5. Run the Server

This cell blocks while the server runs. The tunnel URL above is live.  
Visit `levelupwrestlingapp.com/hs/loop` and click **Start**.

In [ ]:
import subprocess, sys, os

os.chdir('/content/levelup/python')

# Add python dir to path so broken_contacts module is importable
sys.path.insert(0, '/content/levelup/python')
os.environ['PYTHONPATH'] = '/content/levelup/python'

print(f'Starting FastAPI server on port 8100...')
print(f'Tunnel active — Vercel frontend can now connect.\n')

!cd /content/levelup/python && python -m uvicorn server:app --host 0.0.0.0 --port 8100

## 6. (Optional) Test the Server

In [ ]:
# Run in a separate cell while the server is running
# (open a new code cell, the server cell above will keep running)
import requests
r = requests.get('http://localhost:8100/health')
print(r.json())